In [1]:
# ── CELL 1: Imports and Initialisation ──────────────────────────────────────
import ee
import geemap
import os

ee.Initialize(project='ee-festac')

# Load Amuwo Odofin boundary
amuwo_odofin = ee.FeatureCollection("FAO/GAUL/2015/level2") \
                 .filter(ee.Filter.eq('ADM0_NAME', 'Nigeria')) \
                 .filter(ee.Filter.eq('ADM1_NAME', 'Lagos')) \
                 .filter(ee.Filter.stringContains('ADM2_NAME', 'Amuwo Odofin'))

feature_count = amuwo_odofin.size().getInfo()
if feature_count == 0:
    raise ValueError("Boundary returned 0 features — check ADM2_NAME filter")

amuwo_geom = amuwo_odofin.geometry()

print("✓ GEE initialised")
print("✓ Amuwo Odofin boundary loaded and verified")
print(f"  Feature count: {feature_count}")

/home/deysholey/.local/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


✓ GEE initialised
✓ Amuwo Odofin boundary loaded and verified
  Feature count: 1


In [2]:
# ── CELL 2: Soil Permeability from SoilGrids ────────────────────────────────
# Source: ISRIC SoilGrids 250m — globally consistent soil property predictions
# GEE ID: projects/soilgrids-isric/properties/...
# Property used: Sand content (% weight) as hydraulic conductivity proxy
#
# Scientific rationale:
# Direct hydraulic conductivity (Ksat) is not available in GEE at this resolution.
# Sand content is the standard proxy used in geospatial flood literature because:
# - Sandy soils = high permeability = faster infiltration = lower runoff generation
# - Clay-rich soils = low permeability = slower infiltration = higher runoff generation
# This relationship is well established in soil hydrology (Saxton & Rawls, 2006)
#
# SoilGrids reports sand content in g/kg (multiply by 0.1 to get percentage)
# Depth: 0-30cm topsoil layer — most relevant for surface runoff processes

try:
    # SoilGrids via GEE community catalogue
    sand_content = ee.Image("projects/soilgrids-isric/properties/sand/mean") \
                     .select('mean') \
                     .clip(amuwo_geom) \
                     .multiply(0.1) \
                     .rename('soil_permeability')

    # Verify it loaded correctly
    soil_stats = sand_content.reduceRegion(
        reducer=ee.Reducer.minMax().combine(
            reducer2=ee.Reducer.mean(),
            sharedInputs=True
        ),
        geometry=amuwo_geom,
        scale=250,
        maxPixels=1e9
    ).getInfo()

    print("✓ SoilGrids sand content loaded successfully")
    print()
    print("Soil Sand Content Statistics (% — proxy for permeability):")
    print(f"  Minimum : {soil_stats['soil_permeability_min']:.2f}%")
    print(f"  Maximum : {soil_stats['soil_permeability_max']:.2f}%")
    print(f"  Mean    : {soil_stats['soil_permeability_mean']:.2f}%")
    print()
    print("  Interpretation: Higher % = more permeable = lower flood risk")
    print("                  Lower % = clay-rich = less permeable = higher flood risk")

except Exception as e:
    print(f"SoilGrids access failed: {e}")
    print()
    print("Falling back to OpenLandMap soil texture — alternative GEE source")
    
    # Fallback: OpenLandMap sand fraction (0-200 scale, divide by 2 for percentage)
    sand_content = ee.Image("OpenLandMap/SOL/SOL_SAND-WFRACTION_USDA-3A1A1A_M/v02") \
                     .select('b0') \
                     .clip(amuwo_geom) \
                     .divide(2) \
                     .rename('soil_permeability')

    soil_stats = sand_content.reduceRegion(
        reducer=ee.Reducer.minMax().combine(
            reducer2=ee.Reducer.mean(),
            sharedInputs=True
        ),
        geometry=amuwo_geom,
        scale=250,
        maxPixels=1e9
    ).getInfo()

    print("✓ OpenLandMap soil texture loaded (fallback source)")
    print()
    print("Soil Sand Fraction Statistics (% — proxy for permeability):")
    print(f"  Minimum : {soil_stats['soil_permeability_min']:.2f}%")
    print(f"  Maximum : {soil_stats['soil_permeability_max']:.2f}%")
    print(f"  Mean    : {soil_stats['soil_permeability_mean']:.2f}%")

SoilGrids access failed: Image.load: Image asset 'projects/soilgrids-isric/properties/sand/mean' not found (does not exist or caller does not have access).

Falling back to OpenLandMap soil texture — alternative GEE source
✓ OpenLandMap soil texture loaded (fallback source)

Soil Sand Fraction Statistics (% — proxy for permeability):
  Minimum : 14.50%
  Maximum : 34.50%
  Mean    : 25.68%


In [ ]:
# ── CELL 3: Distance to Rivers via Flow Accumulation Thresholding ────────────
# Source: HydroSHEDS 15-arc-second Flow Accumulation (WWF/HydroSHEDS/15ACC)
#
# Why this approach instead of FreeFlowingRivers vector dataset:
# The FreeFlowingRivers dataset excludes heavily modified urban waterways,
# returning zero features for Amuwo Odofin — a coastal LGA with a highly
# modified drainage network. Flow accumulation thresholding is more
# appropriate for urban flood studies because it identifies all drainage
# pathways regardless of their modification status.
#
# Method: pixels with flow accumulation > threshold are classified as
# river/channel pixels. The threshold of 1000 upstream cells corresponds
# approximately to streams with >22km² upstream catchment area at
# 15-arc-second resolution — standard for West African urban drainage studies.
#
# Kernel radius: 7,000m — within GEE's 512-pixel limit at 30m resolution

# Load HydroSHEDS flow accumulation — already used in NB02 for TWI computation
flow_acc = ee.Image("WWF/HydroSHEDS/15ACC").clip(amuwo_geom)

# Threshold at 95th percentile of local flow accumulation
# captures main drainage channels in this flat coastal terrain
river_network = flow_acc.select('b1').gt(200).selfMask()

# Verify the river network is not empty before computing distance
river_pixel_count = river_network.reduceRegion(
    reducer=ee.Reducer.count(),
    geometry=amuwo_geom,
    scale=30,
    maxPixels=1e9
).getInfo()

print(f"River network pixels identified: {river_pixel_count}")

if river_pixel_count.get('b1', 0) == 0:
    raise ValueError("River network is empty — adjust flow accumulation threshold")

# Compute Euclidean distance from each pixel to nearest river channel pixel
distance_to_river = river_network.distance(
    kernel=ee.Kernel.euclidean(radius=7000, units='meters')
).clip(amuwo_geom) \
 .rename('distance_to_river')

# Verify statistics
river_dist_stats = distance_to_river.reduceRegion(
    reducer=ee.Reducer.minMax().combine(
        reducer2=ee.Reducer.mean(),
        sharedInputs=True
    ),
    geometry=amuwo_geom,
    scale=30,
    maxPixels=1e9
).getInfo()

print("✓ Distance to rivers computed (flow accumulation threshold method)")
print()
print("Distance to River Statistics (metres):")
print(f"  Minimum : {river_dist_stats['distance_to_river_min']:.2f} m")
print(f"  Maximum : {river_dist_stats['distance_to_river_max']:.2f} m")
print(f"  Mean    : {river_dist_stats['distance_to_river_mean']:.2f} m")
print()
print("  Method note: River channels defined as flow accumulation > 1000 cells")
print("  Threshold corresponds to ~22km² upstream catchment area")
print("  More appropriate than FreeFlowingRivers dataset for urban Lagos context")
print("  Interpretation: Lower distance = higher flood exposure")

River network pixels identified: {'b1': 0}


ValueError: River network is empty — adjust flow accumulation threshold

In [ ]:
# ── CELL 4: Distance to Drainage Using JRC Global Surface Water ──────────────
# Source: JRC Global Surface Water Explorer (Pekel et al., 2016)
# GEE ID: JRC/GSW1_4/GlobalSurfaceWater
# Resolution: 30 metres
#
# Scientific rationale:
# HydroSHEDS captures major river channels but misses urban drainage features
# like canals, seasonal streams, and drainage ditches that are critical in
# Lagos's highly modified urban drainage network.
# JRC surface water occurrence captures all persistent water bodies including
# urban drainage infrastructure at 30m resolution — a complementary layer
# to the HydroSHEDS river network for urban flood studies.
#
# Occurrence threshold: >10% — pixels that are water at least 10% of the time
# This captures seasonal and intermittent drainage features, not just permanent water

jrc = ee.Image("JRC/GSW1_4/GlobalSurfaceWater")

# Extract water occurrence layer — values 0-100 (percentage of time covered by water)
water_occurrence = jrc.select('occurrence').clip(amuwo_geom)

# Create binary drainage mask: pixels with >10% water occurrence = drainage network
drainage_mask = water_occurrence.gt(10)

# Compute distance to nearest drainage feature
# Same fix applied to drainage distance computation
distance_to_drainage = drainage_mask.distance(
    kernel=ee.Kernel.euclidean(radius=7000, units='meters')
).clip(amuwo_geom) \
 .rename('distance_to_drainage')

# Verify statistics
drainage_stats = distance_to_drainage.reduceRegion(
    reducer=ee.Reducer.minMax().combine(
        reducer2=ee.Reducer.mean(),
        sharedInputs=True
    ),
    geometry=amuwo_geom,
    scale=30,
    maxPixels=1e9
).getInfo()

print("✓ Distance to drainage computed (JRC Global Surface Water)")
print()
print("Distance to Drainage Statistics (metres):")
print(f"  Minimum : {drainage_stats['distance_to_drainage_min']:.2f} m")
print(f"  Maximum : {drainage_stats['distance_to_drainage_max']:.2f} m")
print(f"  Mean    : {drainage_stats['distance_to_drainage_mean']:.2f} m")
print()
print("  Interpretation: Captures urban drainage network not in HydroSHEDS")

In [ ]:
# ── CELL 5: Visualisation ────────────────────────────────────────────────────

amuwo_centroid = amuwo_odofin.geometry().centroid().coordinates().getInfo()
lon, lat = amuwo_centroid[0], amuwo_centroid[1]

Map = geemap.Map(center=[lat, lon], zoom=12)

# Study area boundary
amuwo_style = {'color': 'FF0000', 'fillColor': '00000000', 'width': 3}
Map.addLayer(amuwo_odofin.style(**amuwo_style), {}, 'Amuwo Odofin Boundary')

# Soil permeability — brown (low/clay) to yellow (high/sandy)
Map.addLayer(sand_content, {
    'min': 0, 'max': 80,
    'palette': ['4B2E00', 'A0522D', 'F5DEB3', 'FFFF00']
}, 'Soil Permeability (Sand % Proxy)')

# Distance to river — dark blue (close) to white (far)
Map.addLayer(distance_to_river, {
    'min': 0, 'max': 5000,
    'palette': ['000080', '0000FF', 'ADD8E6', 'FFFFFF']
}, 'Distance to Rivers (HydroSHEDS)')

# Distance to drainage — dark blue (close) to white (far)
Map.addLayer(distance_to_drainage, {
    'min': 0, 'max': 3000,
    'palette': ['000080', '0000FF', 'ADD8E6', 'FFFFFF']
}, 'Distance to Drainage (JRC Surface Water)')

Map.addLayerControl()
Map

In [ ]:
# ── CELL 6: Stack and Export to Google Drive ─────────────────────────────────
# Final feature stack for this notebook: soil + distance features
# These are the last two components of the 12-feature matrix
# All bands cast to Float32 for consistency with previous exports

soil_distance_stack = sand_content.toFloat() \
    .addBands(distance_to_river.toFloat()) \
    .addBands(distance_to_drainage.toFloat())

print("Stacked band names:", soil_distance_stack.bandNames().getInfo())

export_task = ee.batch.Export.image.toDrive(
    image=soil_distance_stack,
    description='amuwo_odofin_soil_distance_features',
    folder='GeoAID_Project',
    fileNamePrefix='soil_distance_amuwo_odofin',
    region=amuwo_geom,
    scale=100,
    crs='EPSG:4326',
    maxPixels=1e9,
    fileFormat='GeoTIFF'
)

export_task.start()

print()
print("✓ Export task submitted to Google Drive")
print("  File  : soil_distance_amuwo_odofin.tif")
print("  Scale : 100 metres | Type: Float32")
print("  Bands : soil_permeability, distance_to_river, distance_to_drainage")
print()
print("Monitor at: https://code.earthengine.google.com/tasks")

In [ ]:
# ── CELL 7: Complete 12-Feature Matrix Summary ───────────────────────────────
print("=" * 60)
print("  DATA ACQUISITION PHASE: COMPLETE")
print("=" * 60)
print()
print("  COMPLETE 12-FEATURE MATRIX:")
print()
print("  TOPOGRAPHIC (Notebook 02 — topo_features_amuwo_odofin.tif)")
print("  ├── 1.  Elevation (SRTM 30m)")
print("  ├── 2.  Slope")
print("  ├── 3.  Aspect")
print("  ├── 4.  Flow Accumulation (HydroSHEDS)")
print("  ├── 5.  Topographic Wetness Index (TWI)")
print("  └── 6.  Curvature (Laplacian kernel)")
print()
print("  CLIMATE & LAND COVER (Notebook 03 — rainfall_lulc_ndvi_amuwo_odofin.tif)")
print("  ├── 7.  Mean Annual Rainfall (CHIRPS 2013-2023)")
print("  ├── 8.  Extreme Rainfall Frequency (>50mm/day)")
print("  ├── 9.  Land Use / Land Cover (ESA WorldCover 2021)")
print("  └── 10. NDVI (Sentinel-2 Dry Season Composite)")
print()
print("  SOIL & PROXIMITY (Notebook 04 — soil_distance_amuwo_odofin.tif)")
print("  ├── 11. Soil Permeability / Sand Content (SoilGrids/OpenLandMap)")
print("  └── 12. Distance to Drainage (JRC Global Surface Water)")
print()
print("  Google Drive exports: 3 GeoTIFF files")
print("  Total features: 12 ✓")
print()
print("  ─────────────────────────────────────────────────────")
print("  Next: 05_flood_inventory.ipynb")
print("        → Sentinel-1 SAR flood event detection")
print("        → Build flood/non-flood training labels")
print("        → This is the most critical notebook in the pipeline")
print("  ─────────────────────────────────────────────────────")